# Pattern 6: Human-in-the-Loop (Approval Gate)

This notebook pauses the agent before the model step so a human can approve or edit input.
It uses static interruption with `interrupt_before=["model"]`.

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0)

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    interrupt_before=["model"],
)

In [ ]:
config = {"configurable": {"thread_id": "approval-demo"}}

# First invoke pauses before model execution and returns interruption state.
paused = agent.invoke(
    {"messages": [{"role": "user", "content": "Draft a short email asking for a meeting tomorrow."}]},
    config=config,
)

print("Interrupted:", "__interrupt__" in paused)
print("Next step:", paused.get("next"))

In [ ]:
# Human review point: optionally replace None with a new input dict to modify the request.
approved_result = agent.invoke(None, config=config)
approved_result["messages"][-1].content

## Usage
For sensitive workflows, require explicit approval before expensive or risky model/tool actions.